In [ ]:
import heapq
from copy import deepcopy

class Puzzle15:
    def __init__(self, initial, goal=None):
        self.initial = initial
        self.goal = goal or [[1, 2, 3, 4],
                            [5, 6, 7, 8],
                            [9, 10, 11, 12],
                            [13, 14, 15, 0]]
        self.n = 4

    def find_blank(self, state):
        for i in range(self.n):
            for j in range(self.n):
                if state[i][j] == 0:
                    return i, j
        return None

    def get_heuristic(self, state):
        """Heuristic: sum of Manhattan distances"""
        distance = 0
        for i in range(self.n):
            for j in range(self.n):
                if state[i][j] != 0:
                    goal_i = (state[i][j] - 1) // self.n
                    goal_j = (state[i][j] - 1) % self.n
                    distance += abs(i - goal_i) + abs(j - goal_j)
        return distance

    def get_neighbors(self, state):
        neighbors = []
        blank_i, blank_j = self.find_blank(state)

        moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # up, down, left, right

        for di, dj in moves:
            new_i, new_j = blank_i + di, blank_j + dj
            if 0 <= new_i < self.n and 0 <= new_j < self.n:
                new_state = deepcopy(state)
                new_state[blank_i][blank_j], new_state[new_i][new_j] = \
                    new_state[new_i][new_j], new_state[blank_i][blank_j]
                neighbors.append(new_state)

        return neighbors

    def state_to_tuple(self, state):
        return tuple(tuple(row) for row in state)

    def solve_akt(self):
        """A* algorithm using AKT (A* with heuristic)"""
        start_node = {
            'state': self.initial,
            'g': 0,
            'h': self.get_heuristic(self.initial),
            'parent': None,
            'action': None
        }
        start_node['f'] = start_node['g'] + start_node['h']

        open_set = [(start_node['f'], id(start_node), start_node)]
        closed_set = set()
        g_values = {self.state_to_tuple(self.initial): 0}

        while open_set:
            _, _, current = heapq.heappop(open_set)
            current_tuple = self.state_to_tuple(current['state'])

            if current_tuple in closed_set:
                continue

            if current['state'] == self.goal:
                return self.reconstruct_path(current)

            closed_set.add(current_tuple)

            for neighbor_state in self.get_neighbors(current['state']):
                neighbor_tuple = self.state_to_tuple(neighbor_state)

                if neighbor_tuple in closed_set:
                    continue

                new_g = current['g'] + 1

                if neighbor_tuple not in g_values or new_g < g_values[neighbor_tuple]:
                    g_values[neighbor_tuple] = new_g
                    neighbor_node = {
                        'state': neighbor_state,
                        'g': new_g,
                        'h': self.get_heuristic(neighbor_state),
                        'parent': current,
                        'action': None
                    }
                    neighbor_node['f'] = neighbor_node['g'] + neighbor_node['h']
                    heapq.heappush(open_set, (neighbor_node['f'], id(neighbor_node), neighbor_node))

        return None

    def reconstruct_path(self, node):
        path = []
        while node:
            path.append(node['state'])
            node = node['parent']
        return list(reversed(path))

    def print_solution(self, path):
        if not path:
            print("No solution found!")
            return

        print(f"Solution found in {len(path) - 1} moves!")
        for i, state in enumerate(path):
            print(f"\nStep {i}:")
            for row in state:
                print(' '.join(f'{x:2d}' for x in row))
        print(f"\nTotal moves: {len(path) - 1}")

# Test
if __name__ == "__main__":
    initial_state = [
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 0, 11],
        [13, 14, 15, 12]
    ]

    puzzle = Puzzle15(initial_state)
    path = puzzle.solve_akt()
    puzzle.print_solution(path)

Solution found in 2 moves!

Step 0:
 1  2  3  4
 5  6  7  8
 9 10  0 11
13 14 15 12

Step 1:
 1  2  3  4
 5  6  7  8
 9 10 11  0
13 14 15 12

Step 2:
 1  2  3  4
 5  6  7  8
 9 10 11 12
13 14 15  0

Total moves: 2


In [ ]:
import heapq

class Graph:
    def __init__(self):
        self.graph = {}

    def add_edge(self, u, v, weight):
        if u not in self.graph:
            self.graph[u] = []
        if v not in self.graph:
            self.graph[v] = []
        self.graph[u].append((v, weight))
        self.graph[v].append((u, weight))

    def heuristic(self, node, goal):
        """Simple heuristic - can be customized based on problem"""
        # For demonstration, using 0 (becomes Dijkstra)
        # In practice, you'd use Euclidean distance or domain-specific heuristic
        return 0

    def a_star_search(self, start, goal):
        """
        A* algorithm to find shortest path between start and goal
        Returns: path and total cost
        """
        if start not in self.graph or goal not in self.graph:
            return None, float('inf')

        # Priority queue: (f_score, counter, node)
        open_set = [(0, 0, start)]
        counter = 1

        # Track g_score (cost from start to node)
        g_score = {node: float('inf') for node in self.graph}
        g_score[start] = 0

        # Track f_score (g_score + heuristic)
        f_score = {node: float('inf') for node in self.graph}
        f_score[start] = self.heuristic(start, goal)

        # Track parent nodes to reconstruct path
        came_from = {}

        # Track nodes in open set
        open_set_nodes = {start}

        while open_set:
            current_f, _, current = heapq.heappop(open_set)

            if current in open_set_nodes:
                open_set_nodes.remove(current)

            # Goal reached
            if current == goal:
                path = self.reconstruct_path(came_from, current)
                return path, g_score[current]

            # Explore neighbors
            for neighbor, weight in self.graph.get(current, []):
                tentative_g = g_score[current] + weight

                if tentative_g < g_score[neighbor]:
                    came_from[neighbor] = current
                    g_score[neighbor] = tentative_g
                    f_score[neighbor] = tentative_g + self.heuristic(neighbor, goal)

                    if neighbor not in open_set_nodes:
                        heapq.heappush(open_set, (f_score[neighbor], counter, neighbor))
                        open_set_nodes.add(neighbor)
                        counter += 1

        return None, float('inf')

    def reconstruct_path(self, came_from, current):
        path = [current]
        while current in came_from:
            current = came_from[current]
            path.append(current)
        return path[::-1]

    def print_path(self, path, cost):
        if not path:
            print("No path found!")
            return

        print(f"Shortest path from {path[0]} to {path[-1]}:")
        print(" -> ".join(str(node) for node in path))
        print(f"Total cost: {cost}")

# Test
if __name__ == "__main__":
    # Create a sample graph
    g = Graph()
    g.add_edge('A', 'B', 4)
    g.add_edge('A', 'C', 2)
    g.add_edge('B', 'C', 1)
    g.add_edge('B', 'D', 5)
    g.add_edge('C', 'D', 8)
    g.add_edge('C', 'E', 10)
    g.add_edge('D', 'E', 2)
    g.add_edge('D', 'F', 6)
    g.add_edge('E', 'F', 3)

    # Find path from A to F
    path, cost = g.a_star_search('A', 'F')
    g.print_path(path, cost)

Shortest path from A to F:
A -> C -> B -> D -> E -> F
Total cost: 13
